<div align="center">

  <a href="https://grayboxtech.github.io/weightslab/latest/index.html" target="_blank">
    <img width="100%" src="https://raw.githubusercontent.com/GrayboxTech/.github/main/profile/weightslab-banner-dark.png" alt="WeightsLab banner"></a>

  <a href="https://github.com/GrayboxTech/weightslab/blob/main/LICENSE"><img src="https://img.shields.io/badge/License-Apache%202.0-blue.svg" alt="License"></a>
  <a href="https://github.com/GrayboxTech/weightslab/stargazers"><img src="https://img.shields.io/github/stars/GrayboxTech/weightslab?style=flat&color=5865F2" alt="Stars"></a>
  <a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/v/weightslab?style=flat&color=5865F2&logo=pypi&logoColor=white" alt="Version"></a>
  <br>
  <a href="https://colab.research.google.com/github/GrayboxTech/weightslab/blob/main/weightslab/examples/Notebooks/Ultralytics/wl-how-to-train-ultralytics-yolo-on-brain-tumor-detection-dataset.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open WeightsLab In Colab"></a>

  Train an <a href="https://github.com/ultralytics/ultralytics">Ultralytics YOLO</a> detector on the MRI <b>brain-tumor</b> dataset and stream every sample, prediction and metric live into <a href="https://github.com/GrayboxTech/weightslab">Weights Studio</a> to <b>inspect, edit and evolve</b> your run.</div>

# Brain Tumor Detection (MRI) · Ultralytics YOLO + WeightsLab

This notebook trains a YOLO detector on the [brain-tumor](https://docs.ultralytics.com/datasets/detect/brain-tumor/) MRI dataset **through the WeightsLab training pipeline**.

Instead of Ultralytics' one-shot `train`/`predict` flow, we hand training to `WLAwareTrainer`. It wraps the train/val dataloaders, logs a **per-sample** signal for every image (loss, prediction stats, tags) and ships live prediction overlays to **Weights Studio** — where you inspect the data grid, remove/weight samples, edit the model and resume, all while training runs.

**Dataset** — 893 training images and 223 validation images, each with detection annotations. Classes: `0: negative`, `1: positive`. Ultralytics auto-downloads it (~4 MB) the first time you train.

## 1 · Setup

Install WeightsLab (which pulls in Ultralytics) and verify the environment.

In [6]:
%pip install setuptools wheel

In [7]:
%pip install git+https://github.com/GrayboxTech/weightslab.git

  Cloning https://github.com/GrayboxTech/weightslab.git (to revision landingcollab) to /tmp/pip-req-build-z4ba7cx9
  Running command git clone --filter=blob:none --quiet https://github.com/GrayboxTech/weightslab.git /tmp/pip-req-build-z4ba7cx9
  Running command git checkout -b landingcollab --track origin/landingcollab
  Switched to a new branch 'landingcollab'
  Branch 'landingcollab' set up to track remote branch 'landingcollab' from 'origin'.
  Resolved https://github.com/GrayboxTech/weightslab.git to commit 3f1cf143bc93e67df43adc71ab831fa71477e0d2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for weightslab: filename=weightslab-1.3.3.dev8-py3-none-any.whl size=2827715 sha256=5c4e7c14bbf95a48b9a8c86b5a8c744af67e1eb395197cbe79d3e443c21cf86e
  Stored in directory: /tmp/pip-ephem-wheel-cache-__zwgf7x/wheels/4d/c7/cd/817c2745790555e7b6c532f5b6055d47567f9416fdfc65879b
Successfully bui

In [1]:
# Install WeightsLab. Colab already ships torch, torchvision, numpy, scikit-learn
# and Pillow, so nothing extra is needed here.
# %pip install weightslab
# %pip install --pre --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ "weightslab==1.3.3.dev8"
%pip install ultralytics

import ultralytics
ultralytics.checks()

import weightslab as wl

Ultralytics 8.4.96 🚀 Python-3.12.13 torch-2.9.0+cu128 CPU (Intel Xeon CPU @ 2.20GHz)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 30.5/107.7 GB disk)
16/07/2026-09:29:29.683 INFO:root:setup_logging: WeightsLab logging initialized - Log file: /tmp/tmp6meuzqdz/weightslab_logs/weightslab_20260716_092929.log
16/07/2026-09:29:29.685 INFO:weightslab:<module>: WeightsLab package initialized - Log level: INFO, Log to file: True
16/07/2026-09:29:29.686 INFO:weightslab:<module>: 
╭─────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                     │
│  $$\      $$\           $$\           $$\         $$\               $$\                 $$\         │
│  $$ | $\  $$ |          \__|          $$ |        $$ |              $$ |                $$ |        │
│  $$ |$$$\ $$ | $$$$$$\  $$\  $$$$$$\  $$$$$$$\  $$$$$$\    $$$$$$$\ $$ |       $$$$$$\  $$$$$$$\    │
│  $$ $$ $$

## 2 · Dataset

The dataset is described by a small YAML file that Ultralytics resolves and downloads automatically — no manual setup needed. For reference, `brain-tumor.yaml` looks like this:

```yaml
path: brain-tumor      # dataset root dir (auto-downloaded)
train: images/train    # 893 images
val: images/val        # 223 images
names:
  0: negative
  1: positive
```

To train on **your own** data, point `data` (in the next cell) at your own `data.yaml` in YOLO format.

## 3 · Train through WeightsLab

We register the run's hyperparameters with `wl.watch_or_edit(...)` (so they become live-editable from Studio), start the backend services with `wl.serve()`, then hand training to `WLAwareTrainer` via the standard `YOLO(...).train(trainer=...)` entry point.

> **Connect Weights Studio** — run `weightslab ui launch` locally (opens `http://localhost:5173`) *before or during* training to watch the run live. On **Colab**, serve over a public tunnel instead: use `wl.serve(serving_grpc=True, serving_bore=True)` and connect your local Studio with `weightslab tunnel bore.pub:PORT` (the port is printed when serving starts).

Data augmentations are disabled so each sample maps cleanly to its ground truth in the grid.

In [2]:
import os
os.environ.setdefault("WL_PRELOAD_IMAGE_OVERVIEW", "0")
os.environ.setdefault("WEIGHTSLAB_LOG_LEVEL", "WARNING")

import warnings
warnings.filterwarnings("ignore")

from weightslab.integrations.ultralytics import WLAwareTrainer
from ultralytics import YOLO

# Hyperparameters registered with WeightsLab -> live-editable from Weights Studio.
cfg = {
    "model": "yolo11n.pt",       # pretrained checkpoint
    "data": "/content/datasets/brain-tumor/brain-tumor.yaml",  # Ultralytics auto-downloads (~4 MB)
    "image_size": 640,
    "epochs": 10,
    # Per-sample prediction overlays streamed to Studio (NMS on the train set).
    "signals_cfg": {
        "train_nms": {"conf_thres": 0.25, "iou_thres": 0.45, "max_nms": 7},
    },
}
wl.watch_or_edit(cfg, flag="hyperparameters", defaults=cfg, poll_interval=1.0)

# Start the WeightsLab backend services that Weights Studio connects to.
wl.serve(serving_grpc=True)
#   Colab -> local Studio:  wl.serve(serving_grpc=True, serving_bore=True)

wl.start_training()  # equivalent to pressing "play" in Studio

# Read back the (now live) hyperparameters and hand training to WLAwareTrainer.
model = YOLO(cfg["model"])
results = model.train(
    trainer=WLAwareTrainer,
    data=str(cfg["data"]),
    imgsz=cfg["image_size"],
    epochs=int(cfg["epochs"]),
    project="./wl_logs", name="brain-tumor",  # -> WL log_dir/name
    workers=0,          # WL invariant (parent-process uid counter)
    optimizer="SGD", lr0=0.001, amp=False,
    device='cpu',
    # All augmentations off for a clean sample <-> ground-truth association.
    mosaic=0.0, mixup=0.0, copy_paste=0.0,
    hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,
    degrees=0.0, translate=0.0, scale=0.0, shear=0.0, perspective=0.0,
    flipud=0.0, fliplr=0.0, erasing=0.0, auto_augment=None,
)

16/07/2026-09:29:52.825 INFO:weightslab.src:watch_or_edit: LoggerQueue for experiment history has been initialized and registered.
16/07/2026-09:29:52.828 INFO:weightslab.components.checkpoint_manager:__init__: CheckpointManager initialized at /tmp/tmp3abnx3ay
16/07/2026-09:29:52.829 INFO:weightslab.src:watch_or_edit: Registered new checkpoint manager in ledger
16/07/2026-09:29:52.830 INFO:root:set_log_directory: Log file moved from /tmp/tmp6meuzqdz/weightslab_logs/weightslab_20260716_092929.log to /tmp/tmp3abnx3ay/weightslab_20260716_092929.log
16/07/2026-09:29:52.832 INFO:root:set_log_directory: Log directory updated to: /tmp/tmp3abnx3ay
16/07/2026-09:29:52.833 INFO:root:set_log_directory: Log file: /tmp/tmp3abnx3ay/weightslab_20260716_092929.log
16/07/2026-09:29:52.835 INFO:weightslab.trainer.trainer_services:_run_security_preflight: 

# #######################################
# #######################################
[gRPC] Security preflight checks:
	TLS: DISABLED (unencrypted tra

Initializing ledger for split 'train_loader': 100%|██████████| 893/893 [00:00<00:00, 13970.17it/s]

16/07/2026-09:29:54.600 INFO:weightslab.data.dataframe_manager:register_split: [LedgeredDataFrameManager] Registering split 'train_loader' with 893 samples.
16/07/2026-09:29:54.615 INFO:weightslab.data.dataframe_manager:register_split: [LedgeredDataFrameManager] After annotation expansion: 893 samples → 984 annotation rows.
16/07/2026-09:29:54.616 INFO:weightslab.data.h5_array_store:__init__: [H5ArrayStore] Initialized with cache limit: 2048MB
16/07/2026-09:29:54.636 INFO:weightslab.components.checkpoint_manager:load_checkpoint: Loading checkpoint 1b111ac400000000...
16/07/2026-09:29:54.637 INFO:weightslab.components.checkpoint_manager:load_checkpoint:  Target: HP=1b111ac4 MODEL=00000000 DATA=00000000
16/07/2026-09:29:54.638 INFO:weightslab.components.checkpoint_manager:load_checkpoint:  Current: HP=1b111ac4 MODEL=00000000 DATA=00000000
16/07/2026-09:29:54.639 INFO:weightslab.components.checkpoint_manager:load_checkpoint:  [-] Model architecture unchanged, using current model
16/07/202


Initializing ledger for split 'val_loader': 100%|██████████| 223/223 [00:00<00:00, 5362.24it/s]

16/07/2026-09:29:54.708 INFO:weightslab.data.dataframe_manager:register_split: [LedgeredDataFrameManager] Registering split 'val_loader' with 223 samples.
16/07/2026-09:29:54.714 INFO:weightslab.data.dataframe_manager:register_split: [LedgeredDataFrameManager] After annotation expansion: 223 samples → 257 annotation rows.


16/07/2026-09:29:54.734 INFO:weightslab.components.checkpoint_manager:load_checkpoint: Loading checkpoint 1b111ac400000000...
16/07/2026-09:29:54.736 INFO:weightslab.components.checkpoint_manager:load_checkpoint:  Target: HP=1b111ac4 MODEL=00000000 DATA=00000000
16/07/2026-09:29:54.739 INFO:weightslab.components.checkpoint_manager:load_checkpoint:  Current: HP=1b111ac4 MODEL=00000000 DATA=00000000
16/07/2026-09:29:54.740 INFO:weightslab.components.checkpoint_manager:load_checkpoint:  [-] Model architecture unchanged, using current model
16/07/2026-09:29:54.742 INFO:weightslab.components.checkpoint_manager:load_checkpoint:  [-] Config unchanged, using current config
16/07/2026-09:29:54.743 WARNING:weightslab.components.checkpoint_manager:load_checkpoint:  [WARNING] Data snapshot file not found: /tmp/tmp3abnx3ay/checkpoints/data/00000000/00000000_data_snapshot.json
16/07/2026-09:29:54.746 INFO:weightslab.components.checkpoint_manager:load_checkpoint: Loaded components: set()
optimizer: S

KeyboardInterrupt: 

## 4 · Explore the data grid

The table below is exactly the data behind Weights Studio's **grid explorer**: one row per sample, keyed by split (`origin`) and sample id, with the per-sample loss, prediction stats and tags accumulated during training. This is the ledger that Studio reads from — here we render it inline with pandas.

The image thumbnails and bounding-box overlays themselves are rendered by **Weights Studio** (which streams full previews over gRPC); the link below opens the interactive grid there.

In [3]:
import pandas as pd
from IPython.display import display, HTML
from weightslab.backend.ledgers import get_dataframe

# `get_df_view()` is the same per-sample DataFrame that powers Studio's grid.
dfm = get_dataframe()
grid = dfm.get_df_view() if hasattr(dfm, "get_df_view") else pd.DataFrame()
print(f"{len(grid)} samples tracked")
display(grid.head(50))

# Open the live, interactive grid (with image + bbox previews) in Weights Studio.
# Studio must be running locally (`weightslab start`).
display(HTML(
    '<a href="http://localhost:5173" target="_blank" '
    'style="font-size:15px">&#128279; Open the full grid in Weights Studio</a>'
))

1241 samples tracked


prediction prediction_raw  \
sample_id annotation_id                             
0         0                   None           None   
1         0                   None           None   
2         0                   None           None   
          1                   None           None   
          2                   None           None   
3         0                   None           None   
4         0                   None           None   
5         0                   None           None   
6         0                   None           None   
7         0                   None           None   
8         0                   None           None   
9         0                   None           None   
10        0                   None           None   
11        0                   None           None   
12        0                   None           None   
13        0                   None           None   
14        0                   None           None   
          1                   None           None   
          2                   None           None   
15        0                   None           None   
          1                   None           None   
          2                   None           None   
16        0                   None           None   
          1                   None           None   
          2                   None           None   
17        0                   None           None   
18        0                   None           None   
19        0                   None           None   
20        0                   None           None   
          1                   None           None   
          2                   None           None   
          3                   None           None   
21        0                   None           None   
22        0                   None           None   
23        0                   None           None   
24        0                   None           None   
25        0                   None           None   
          1                   None           None   
          2                   None           None   
26        0                   None           None   
27        0                   None           None   
28        0                   None           None   
29        0                   None           None   
30        0                   None           None   
31        0                   None           None   
32        0                   None           None   
33        0                   None           None   
34        0                   None           None   
35        0                   None           None   
36        0                   None           None   

                                                                    target  \
sample_id annotation_id                                                      
0         0              [0.2335685, 0.254695, 0.4553995, 0.43075103, 1...   
1         0              [0.251174, 0.24061048, 0.44366202, 0.4307515, ...   
2         0                                                           None   
          1              [0.523474, 0.3978875, 0.634976, 0.48122054, 1....   
          2              [0.40610352, 0.415493, 0.5234745, 0.522301, 1....   
3         0              [0.403756, 0.3826295, 0.637324, 0.5152585, 1.0...   
4         0              [0.4436615, 0.3814555, 0.5938965, 0.4518785, 1...   
5         0              [0.6044605, 0.3990615, 0.67253554, 0.46596247,...   
6         0              [0.410798, 0.4436625, 0.556338, 0.51173747, 1....   
7         0              [0.650235, 0.4166665, 0.73826295, 0.48356748, ...   
8         0              [0.64788747, 0.388498, 0.7582165, 0.49413198, ...   
9         0              [0.6807515, 0.40140802, 0.75704247, 0.45774597...   
10        0              [0.431925, 0.1701875, 0.561033, 0.3180745, 1.0...   
11        0              [0.42018852, 0.34507048, 0.5434275, 0.4941315,...   
12        0        

> **Can clicking a grid row here jump to that sample in Studio?** Not out of the box today. The notebook and Studio are separate processes, and the link above opens Studio at its root. Per-sample deep-linking (e.g. `http://localhost:5173/?sample=<uid>` selecting and scrolling to that row) is a small frontend addition to Weights Studio — the sample `uid` is already the grid's index here, so the notebook side is ready for it once Studio accepts the query param.

## Ultralytics Citation

```bibtex
@dataset{Jocher_Ultralytics_Datasets_2024,
    author = {Jocher, Glenn and Rizwan, Muhammad},
    license = {AGPL-3.0},
    title = {Ultralytics Datasets: Brain-tumor Detection Dataset},
    url = {https://docs.ultralytics.com/datasets/detect/brain-tumor/},
    version = {1.0.0},
    year = {2024}
}
```